# 02F — M1 partial-only Musanga segmentation: six-mosaic ROC-AUC check

Train **only from non-exhaustive tree annotations** (Musanga `Class_ID=1`, other numeric classes non-Musanga); unknown background stays unlabelled. **Every one of the 34 Nadine exhaustive circles is excluded with a 15 m buffer from ALL M1 training chips**, across all MOS mosaics, and is used exclusively for independent assessment. No exhaustive annotation is involved in training.

Six mosaics: Baego, Yoko, Yangambi, **MOS 1–2, MOS 3–4, MOS 5–6**. Evaluate pixel-level ROC-AUC, AP, F1 at 0.5, Brier and mean predicted canopy fraction against all 34 exhaustive circles, separately by site and MOS mosaic. Also report circle-level canopy-fraction R² and MAE as a different quantity.

**Interpretation:** ROC-AUC reflects pixel-ranking/discrimination, whereas MAE of averaged canopy fraction reflects calibration and quantification. High ROC-AUC can coexist with poor fraction estimates.

This is a fresh **M1-only** 25-epoch GPU run with separate checkpoint/cache. Previous experiments are untouched.


## 01 — Input registry (34 Nadine circles, six mosaics)

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02F_M1_partial_only_six_mosaics';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data_nadine34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==34
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':15,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
assert len(mosaics)==6,[(k,v['site']) for k,v in mosaics.items()]
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('M1 partial-only: all 34 exhaustive circles EXCLUDED from training; independent ROC-AUC evaluation')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('Mosaics:',[(name,m['site']) for name,m in mosaics.items()])


## 02 — Load annotations and TinyUNet

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)==1368,(
    'Unexpected reference geometry count; do not train using a partial dataset')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — Strict training exclusion: all 34 exhaustive circles (including MOS cross-mosaic buffers)

In [ ]:
HOLDOUT=set(links.circle_id.astype(int))
DEV=set()
split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
split_df['split']='holdout'
BLOCKED={}
for name,d in geo.items():
    this=circles.to_crs(d['crs'])
    # 15m exclusion across the complete site, not only the mosaic assigned to a circle.
    site_ids=set(links.loc[links.site==d['site'],'circle_id'].astype(int))
    BLOCKED[name]=[g.buffer(15.) for g in this[this.original_row.isin(site_ids)].geometry]
assert len(HOLDOUT)==34
print('All',len(HOLDOUT),'circles excluded from training, across MOS mosaic boundaries.')


## 04 — Indexed label rasterization, partial-only crops, persistent cache

In [ ]:
from rasterio.windows import Window
,from torch.utils.data import Dataset,DataLoader
,candidate=pd.read_csv(REF/'01_candidate_windows.csv')
,all_circle_buffers={}
,for name,d in geo.items():
,    cs=circles.to_crs(d['crs'])
,    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
,    d['dev_circles']=cs[cs.original_row.isin(DEV)]
,    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
,    d['positive_local']=positive.to_crs(d['crs'])
,    d['trees_local']=tree[d['site']].to_crs(d['crs'])
,    d['roi_local']=roi.to_crs(d['crs'])
,
,def burn_window(series,name,win):
,    """Rasterize only polygons intersecting this window (spatial-index accelerated)."""
,    d=geo[name]
,    bb=box(*rasterio.windows.bounds(win,d['transform']))
,    ids=series.sindex.query(bb,predicate='intersects')
,    if len(ids)==0:
,        return np.zeros((int(win.height),int(win.width)),dtype=bool)
,    gs=series.iloc[ids]
,    return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
,        out_shape=(int(win.height),int(win.width)),
,        transform=rasterio.windows.transform(win,d['transform']),
,        fill=0,dtype='uint8').astype(bool)
,
,def rgb_for_window(name,win):
,    with rasterio.open(geo[name]['path']) as ds:
,        raw=ds.read([1,2,3],window=win,masked=True)
,    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
,    a=raw.filled(0).astype('float32')
,    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
,    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255)) # blue Baego nodata
,    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid
,
,def training_crop(rec,model):
,    name=rec.mosaic;d=geo[name]
,    if rec.height<CROP or rec.width<CROP:return None
,    rr=int(rec.row)+(int(rec.height)-CROP)//2
,    cc=int(rec.col)+(int(rec.width)-CROP)//2
,    win=Window(cc,rr,CROP,CROP)
,    bb=box(*rasterio.windows.bounds(win,d['transform']))
,    if any(bb.intersects(g) for g in BLOCKED[name]):return None
,    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
,    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
,    inside=burn_window(d['roi_local'].geometry,name,win)
,    t=d['trees_local']
,    pos_tree=burn_window(t.loc[t.class01==1].geometry,name,win)
,    neg_tree=burn_window(t.loc[t.class01==0].geometry,name,win)
,    if model=='M1':
,        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
,    else:
,        full=burn_window(d['dev_circles'].geometry,name,win)&inside
,        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
,    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
,    target=(pos_ex|(~full&pos_tree)).astype('float32')
,    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
,    img,valid=rgb_for_window(name,win)
,    weight &= valid
,    if int(weight.sum())<250:return None
,    return dict(image=img,target=target,weight=weight.astype('float32'),
,                site=d['site'],kind=rec.kind,mosaic=name)
,
,def make_training_cache(model):
,    selected=[]
,    for (site,kind),grp in candidate.groupby(['site','kind']):
,        if model=='M1' and kind=='exhaustive':continue
,        if model=='M2' and kind=='exhaustive':
,            grp=grp[grp.circle_id.isin(DEV)]
,        limit=MAX_EXHAUSTIVE_PER_SITE if kind=='exhaustive' else MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
,        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
,        selected.extend(subset.itertuples(index=False))
,    result=[]
,    print('M1 candidate windows per mosaic:')
    display(pd.DataFrame([dict(mosaic=k,n=len(g),site=g.site.iloc[0]) for k,g in candidate[candidate.kind=='partial'].groupby('mosaic')]))
    for i,rec in enumerate(selected):
,        chip=training_crop(rec,model)
,        if chip is not None:result.append(chip)
,        if i and i%100==0:print(model,'prepared',i,'/',len(selected),'usable',len(result),flush=True)
,    site_counts = pd.Series([item["site"] for item in result]).value_counts().to_dict()
,    print(f"{model}: usable training chips = {len(result)}", flush=True)
,    print("Training chips by site:", site_counts, flush=True)
,    assert len(result)>15,f'{model}: too few chips'
,    return result
,
,class CropDataset(Dataset):
,    def __init__(self,rows):self.rows=rows
,    def __len__(self):return len(self.rows)
,    def __getitem__(self,i):
,        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
,        k=int(torch.randint(0,4,(1,)).item())
,        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
,        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
,        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
,                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))
,

# Cache only native, non-augmented crops; record source-file metadata and training rules.
import hashlib
CACHE_DIR=OUT/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
def fingerprint():
    deps=[]
    paths=[REF/'01_candidate_windows.csv',
           ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json',
           A['roi'],A['exhaustive_annotations']['complete_roi_boundaries']]
    paths += [A['shared_annotations'][key] for key in sorted(A['shared_annotations'])]
    paths += [v['path'] for v in mosaics.values()]
    for path in paths:
        path=Path(path)
        more=[path.with_suffix(e) for e in ('.shp','.shx','.dbf','.prj','.cpg')
              if path.with_suffix(e).exists()] if path.suffix=='.shp' else [path]
        for p in more:
            deps.append([str(p),p.stat().st_size,p.stat().st_mtime_ns])
    spec=dict(version='M1six_34excluded_center256_v1',dependencies=deps,
              seed=SEED,crop=CROP,holdout_ids=sorted(HOLDOUT),
              max_partial=MAX_PARTIAL_PER_MOSAIC,
              conflict_weight_zero=True)
    return hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
CACHE_KEY=fingerprint()
CACHE_FILE=CACHE_DIR/f'M1_{CACHE_KEY[:20]}.npz'
def get_m1_cache():
    if CACHE_FILE.exists():
        started=time.perf_counter()
        with np.load(CACHE_FILE,allow_pickle=False) as z:
            assert str(z['key'].item())==CACHE_KEY
            rgb=z['rgb'];target=z['target'];weight=z['weight']
            sites=z['site'].astype(str)
            kinds=z['kind'].astype(str)
            assert len(rgb)>15 and rgb.shape[1:]==(CROP,CROP,3)
            result=[dict(image=rgb[i].astype('float32')/255.,
                         target=target[i].astype('float32'),
                         weight=weight[i].astype('float32'),
                         site=str(sites[i]),kind=str(kinds[i]))
                    for i in range(len(rgb))]
        print('M1 CACHE HIT:',len(result),'chips restored in',
              round(time.perf_counter()-started,1),'seconds',flush=True)
        return result
    print('M1 cache miss: preparing source crops once.',flush=True)
    started=time.perf_counter()
    result=make_training_cache('M1')
    temp=CACHE_FILE.with_name(CACHE_FILE.name+'.tmp.npz')
    try:
        np.savez_compressed(temp,key=np.array(CACHE_KEY),
            rgb=np.stack([np.rint(np.clip(r['image'],0,1)*255).astype('uint8') for r in result]),
            target=np.stack([r['target'].astype('uint8') for r in result]),
            weight=np.stack([r['weight'].astype('uint8') for r in result]),
            site=np.array([r['site'] for r in result],dtype='U12'),
            kind=np.array([r['kind'] for r in result],dtype='U16'))
        temp.replace(CACHE_FILE)
    finally:
        temp.unlink(missing_ok=True)
    print('M1 CACHE SAVED:',len(result),'chips in',
          round(time.perf_counter()-started,1),'seconds',flush=True)
    return result


## 05 — Train M1 (25 epochs), AMP, checkpoint reuse

In [ ]:
import time
,from contextlib import nullcontext
,USE_AMP=(device=='cuda')
,# float16 is fast on both T4 and L4; BCE is evaluated in fp32 for stability.
,scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
,print('CUDA mixed precision:',USE_AMP,'| GPU:',torch.cuda.get_device_name(0),flush=True)
,
,def fit_m1():
,    ckpt=OUT/'M1_model.pt'
,    torch.manual_seed(SEED)
,    model=TinyUNet().to(device)
,    if ckpt.exists():
,        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
,        print('Reusing completed M1 partial-only checkpoint:',ckpt,flush=True)
,        return model
,    print('M1: checking persisted partial-only cache...',flush=True)
,    prep=time.perf_counter()
,    cache=get_m1_cache()
,    print(f'M1: {len(cache)} usable training chips; prep {time.perf_counter()-prep:.1f}s',flush=True)
,    loader=DataLoader(CropDataset(cache),batch_size=BATCH,shuffle=True,num_workers=0,
,                      pin_memory=True)
,    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
,    history=[]
,    t0=time.perf_counter()
,    for epoch in range(EPOCHS):
,        model.train();losses=[];t_epoch=time.perf_counter()
,        for step,(x,y,w) in enumerate(loader,1):
,            x=x.to(device,non_blocking=True)
,            y=y.to(device,non_blocking=True)
,            w=w.to(device,non_blocking=True)
,            optim.zero_grad(set_to_none=True)
,            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
,                logits=model(x)
,            loss=masked_loss(logits.float(),y,w)
,            scaler.scale(loss).backward()
,            scaler.step(optim)
,            scaler.update()
,            losses.append(float(loss.detach()))
,        elapsed=time.perf_counter()-t_epoch
,        avg=float(np.mean(losses))
,        history.append(dict(epoch=epoch+1,masked_BCE=avg,seconds=elapsed))
,        pd.DataFrame(history).to_csv(OUT/'M1_loss.csv',index=False)
,        print(f'M1 epoch {epoch+1:02d}/{EPOCHS}: BCE={avg:.4f}, '
,              f'{elapsed:.1f}s, total {(time.perf_counter()-t0)/60:.1f} min',
,              flush=True)
,    torch.save(model.state_dict(),ckpt)
,    print('M1 training complete; checkpoint:',ckpt,flush=True)
,    del cache
,    return model
,
,model1=fit_m1()
,print('M1 ready. Proceed to independent 34-circle ROC-AUC assessment.',flush=True)


## 06 — Independent exhaustive circle predictions (34 circles, no M1 training overlap)

In [ ]:
from scipy.special import expit,logit
,from scipy.optimize import minimize
,from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
,WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
,VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
,assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
,N_BINS=256
,BINS=np.linspace(0,1,N_BINS+1)
,BIN_CENTER=(BINS[:-1]+BINS[1:])/2
,
,def run_all_circles(model,model_name):
,    statfile=OUT/f'{model_name}_quick34_circle_raw.csv'
,    histfile=OUT/f'{model_name}_quick34_histograms.npz'
,    if statfile.exists() and histfile.exists():
,        return pd.read_csv(statfile),dict(np.load(histfile))
,    model.eval();rows=[];histograms={}
,    for _,circle in split_df.iterrows():
,        cid=int(circle.circle_id)
,        name=circle.mosaic
,        d=geo[name]
,        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
,        assert len(windows),f'No evaluation windows for circle {cid}'
,        cg=circles[circles.original_row==cid].to_crs(d['crs'])
,        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
,        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
,        rng=np.random.default_rng(SEED+cid)
,        sample_truth=[];sample_probs=[]
,        for rec in windows.itertuples(index=False):
,            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
,            mask=burn_window(cg.geometry,name,win)
,            mask &= burn_window(d['roi_local'].geometry,name,win)
,            rgb,valid=rgb_for_window(name,win)
,            mask &= valid
,            if not mask.any():continue
,            reference=burn_window(d['positive_local'].geometry,name,win)
,            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
,            with torch.inference_mode():
,                pred=torch.sigmoid(model(x))[0].cpu().numpy()
,            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
,            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
,            sum_sq+=float(np.square(p-y).sum())
,            binary=p>=.5
,            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
,            fn+=int(np.sum((~binary)&(y==1)))
,            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
,            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
,            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
,            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
,            take=rng.choice(len(y),min(100,len(y)),replace=False)
,            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
,        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
,        assert int(counts.sum())==n
,        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
,        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
,                         valid_pixels=n,observed_fraction=100*sumy/n,
,                         predicted_fraction=100*sump/n,
,                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
,                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
,                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
,                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
,        histograms[str(cid)+'_count']=counts
,        histograms[str(cid)+'_positive']=true_counts
,        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
,              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
,    frame=pd.DataFrame(rows)
,    assert frame.circle_id.is_unique
,    assert set(frame.circle_id.astype(int))==set(links.circle_id.astype(int))
,    assert len(frame)==34
,    frame.to_csv(statfile,index=False)
,    np.savez_compressed(histfile,**histograms)
,    return frame,histograms
,
,m1, hist1=run_all_circles(model1,'M1')
,print('M1: all 34 circle predictions saved; ALL are excluded from training.',flush=True)
,


## 07 — Exact binned pixel ROC-AUC/AP by mosaic, site and overall; fraction MAE separate

In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score,r2_score,mean_absolute_error
# Combine 256-bin histograms of *all* valid image/reference pixels rather than
# averaging per-circle sampled AUCs. This evaluates pixel discrimination; the
# histograms provide a ~1/256 probability-resolution approximation to exact AUC.
def binned_metrics(ids):
    cnt=np.sum([hist1[str(int(cid))+'_count'] for cid in ids],axis=0).astype('float64')
    pos=np.sum([hist1[str(int(cid))+'_positive'] for cid in ids],axis=0).astype('float64')
    neg=cnt-pos
    assert (neg>=0).all()
    npos=pos.sum();nneg=neg.sum()
    if npos==0 or nneg==0:return dict(auc=np.nan,ap=np.nan,pixels=int(cnt.sum()))
    # Tie-aware Mann-Whitney: negatives with lower scores + half tied negatives.
    auc=float(np.dot(pos,np.cumsum(neg)-.5*neg)/(npos*nneg))
    # AP uses descending score order and precision at each positive score.
    pr=np.cumsum(pos[::-1]);fp=np.cumsum(neg[::-1])
    ap=float(np.sum((pos[::-1]/npos)*(pr/np.maximum(pr+fp,1))))
    return dict(auc=auc,ap=ap,pixels=int(cnt.sum()),positive_pct=100*npos/(npos+nneg))
report=[]
for site,part in m1.groupby('site'):
    report.append(dict(group_type='site',group=site,circles=len(part),
                       **binned_metrics(part.circle_id),
                       MAE_fraction_pp=float(part.bias_pp.abs().mean())))
for mosaic,part in m1.merge(links[['circle_id','mosaic']],on='circle_id').groupby('mosaic'):
    report.append(dict(group_type='mosaic',group=mosaic,circles=len(part),
                       **binned_metrics(part.circle_id),
                       MAE_fraction_pp=float(part.bias_pp.abs().mean())))
report.append(dict(group_type='overall',group='ALL 34 independent circles',
                   circles=len(m1),**binned_metrics(m1.circle_id),
                   MAE_fraction_pp=float(m1.bias_pp.abs().mean())))
REPORT=pd.DataFrame(report).sort_values(['group_type','group'])
assert m1.circle_id.nunique()==34
assert set(m1.circle_id)==HOLDOUT
print('=== INDEPENDENT M1 PIXEL DISCRIMINATION: 34 exhaustive circles ===')
display(REPORT.round(4))
print('=== CIRCLE-LEVEL FRACTION: ALL 34 (independent) ===')
print('R²',round(r2_score(m1.observed_fraction,m1.predicted_fraction),4),
      '| MAE pp',round(mean_absolute_error(m1.observed_fraction,m1.predicted_fraction),3))
REPORT.to_csv(OUT/'02F_M1_roc_auc_by_site_mosaic.csv',index=False)
m1.to_csv(OUT/'02F_M1_all34_circle_predictions.csv',index=False)


## 08 — Independent per-site fraction scatterplot and experiment contract

In [ ]:
colors={'Baego':'#c27e24','MOS':'#357ebd','Yangambi':'#468c50','Yoko':'#944b99'}
fig,ax=plt.subplots(figsize=(7,6.5),constrained_layout=True)
ax.plot([0,100],[0,100],ls='--',color='gray')
for row in m1.itertuples(index=False):
    ax.scatter(row.observed_fraction,row.predicted_fraction,
               s=45,color=colors[row.site],alpha=.75)
    ax.annotate(str(row.circle_id),
                (row.observed_fraction,row.predicted_fraction),
                xytext=(3,3),textcoords='offset points',fontsize=7)
ax.set(xlim=(0,100),ylim=(0,100),
    xlabel='Nadine observed Musanga canopy fraction (%)',
    ylabel='M1 mean predicted probability (%)',
    title='M1 partial-only: 34 independently held-out circles')
ax.grid(alpha=.2)
fig.savefig(OUT/'02F_M1_all34_fraction_scatter.png',dpi=180,bbox_inches='tight')
display(fig);plt.close(fig)
contract=dict(schema='M1_partial_only_six_mosaics_v1',
    model='TinyUNet M1 partial annotations only',
    exclude_all_34_exhaustive_circles_from_training=True,
    exclusion_buffer_m=15,train_mosaic_files=sorted(mosaics),
    training_labels='Shared partial tree annotations Class_ID 1 vs other numeric; unlabeled unknown',
    evaluation_labels='Nadine exhaustive Musanga polygons vs full circles',
    roc_auc='256-bin pooled probability ROC-AUC; independent pixel labels',
    reference_circles=34,epochs=EPOCHS,checkpoint=str(OUT/'M1_model.pt'),
    caution='ROI and valid-RGB masks apply; high AUC does not imply calibrated fractional cover')
(OUT/'02F_M1_contract.json').write_text(json.dumps(contract,indent=2))
print('Independent 34-circle M1 check completed. Outputs:',OUT)
